# Sizing

So far every `size` and `capacity` has been a number we picked by hand. **`Sizing`** lets the optimizer pick them, trading the cost of building bigger against the cost of operating smaller.

You'll meet **`Sizing`** — a parameter object that turns a fixed `size` (or `capacity`) into a decision variable, optionally with per-MW investment cost.

In [ ]:
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from fluxopt import Carrier, Converter, Effect, Flow, Port, Sizing, Storage, optimize

pio.renderers.default = 'notebook_connected'

## 1. From numbers to `Sizing`

Anywhere you wrote a fixed `size=120` or `capacity=300`, you can write `Sizing(size_min=..., size_max=..., effects_per_size=...)` instead. The fields:

- `size_min` / `size_max` — the bracket the optimizer searches.
- `effects_per_size` — investment cost per unit, charged once. `{'cost': 0.27}` means €0.27 per kW of installed capacity.
- `mandatory=True` (default) — must build something within `[min, max]`. Set `False` to let the optimizer choose not to build at all (introduces a binary).

We use **daily-amortized** rates here so that the 24-hour run is meaningful: €0.27/kW/day for the boiler (~€1000/kW lifetime over 10 years), €0.06/kWh/day for the tank (~€220/kWh over 10 years). T7 will show how multi-period models handle proper investment timing across years.

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]
gas_price = [0.04 if h < 6 or h >= 22 else 0.12 for h in range(n)]
demand = np.array([10, 10, 8, 8, 8, 12, 25, 60, 70, 75, 75, 70, 65, 60, 55, 50, 45, 40, 30, 25, 20, 15, 12, 10])
peak = max(demand)

In [ ]:
result = optimize(
    timesteps=timesteps,
    carriers=[Carrier(id='gas'), Carrier(id='heat')],
    effects=[Effect(id='cost', unit='EUR')],
    ports=[
        Port(
            id='gas_grid',
            imports=[
                Flow(carrier='gas', size=1000, effects_per_flow_hour={'cost': gas_price}),
            ],
        ),
        Port(
            id='workshop',
            exports=[
                Flow(carrier='heat', size=peak, fixed_relative_profile=[d / peak for d in demand]),
            ],
        ),
    ],
    converters=[
        Converter.boiler(
            'boiler',
            thermal_efficiency=0.9,
            fuel_flow=Flow(carrier='gas', size=300),
            thermal_flow=Flow(
                carrier='heat',
                size=Sizing(size_min=20, size_max=200, effects_per_size={'cost': 0.27}),
            ),
        ),
    ],
    storages=[
        Storage(
            id='tank',
            capacity=Sizing(size_min=0, size_max=1000, effects_per_size={'cost': 0.06}),
            charging=Flow(carrier='heat', size=80),
            discharging=Flow(carrier='heat', size=80),
            eta_charge=0.98,
            eta_discharge=0.98,
            relative_loss_per_hour=0.005,
        ),
    ],
    objective='cost',
)
print(f'Total cost (operating + investment): {result.objective:.2f} EUR')

## 2. The chosen sizes

`result.sizes` is a `flow`-indexed array (NaN for fixed-size flows); `result.storage_capacities` is `storage`-indexed. Drop the NaNs to get just the optimized values.

In [ ]:
result.sizes.dropna(dim='flow').to_dataframe(name='size [kW]')

In [ ]:
result.storage_capacities.to_dataframe(name='capacity [kWh]')

In [ ]:
flows = result.flow_rates.to_dataframe('value').reset_index().assign(panel='flow rate (kW)')
level = result.storage_level('tank').to_dataframe('value').reset_index().assign(panel='tank level (kWh)', flow='tank')
df = pd.concat([flows, level], ignore_index=True)
fig = px.line(df, x='time', y='value', color='flow', facet_row='panel', line_shape='hv', height=440)
fig.update_yaxes(matches=None, title='')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(template='plotly_white', margin={'l': 50, 'r': 20, 't': 30, 'b': 20})
fig

The optimizer picks a boiler smaller than the demand peak (75 kW) — it's cheaper to build less boiler and arbitrage across the TOU spread with a generous tank. Push the tank's `effects_per_size` up and the answer flips: the boiler grows toward the peak and the tank shrinks.

## Recap

`Sizing` turns design choices into LP variables. Pair it with `effects_per_size` to capture investment cost; pair `mandatory=False` with `size_min > 0` to give the optimizer a yes/no build choice (introduces a binary).

**Next**: [Multi-period & Investment](07-multi-period.ipynb) — handle multiple periods (years) at once and decide *when* to build, not just *whether*.